# Audio Emotion Detector - Colab Demo

This notebook demonstrates the same functionality as the Hugging Face Space.
Upload an audio clip and see what emotion the AI detects - happy, sad, angry, or neutral.

**Model:** `superb/wav2vec2-base-superb-er` (speech emotion recognition)

The model listens to *how* you say things (pitch, energy, pace), not *what* you say.

In [ ]:
!pip install -q transformers torch librosa

In [ ]:
from transformers import pipeline
from IPython.display import display, HTML, Audio
from google.colab import files
import librosa

classifier = pipeline(
    "audio-classification",
    model="superb/wav2vec2-base-superb-er"
)
print("Model loaded!")

In [ ]:
LABEL_MAP = {
    "neu": "Neutral", "hap": "Happy", "sad": "Sad", "ang": "Angry",
    "neutral": "Neutral", "happy": "Happy", "sad": "Sad", "angry": "Angry",
}

EMOTION_STYLE = {
    "Happy":  {"emoji": "\U0001f60a", "color": "#f1c40f"},
    "Sad":    {"emoji": "\U0001f622", "color": "#3498db"},
    "Angry":  {"emoji": "\U0001f620", "color": "#e74c3c"},
    "Neutral":{"emoji": "\U0001f610", "color": "#95a5a6"},
}

## Option A: Upload an Audio File

Run the cell below and upload a WAV or MP3 file.

In [ ]:
uploaded = files.upload()
audio_filename = list(uploaded.keys())[0]
print(f"Uploaded: {audio_filename}")

# Play the audio
display(Audio(audio_filename))

## Option B: Record Audio in Colab

If you don't have an audio file, run this cell to record from your microphone.

In [ ]:
# JavaScript-based audio recorder for Colab
from google.colab import output
from base64 import b64decode
from IPython.display import Javascript

RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});

var record = async function(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream, {mimeType: 'audio/webm'});
  let chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  document.querySelector('#recording-status').textContent = 'Recording...';
  await sleep(seconds * 1000);
  recorder.stop();
  document.querySelector('#recording-status').textContent = 'Done!';
  await new Promise(r => recorder.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  return await b2text(new Blob(chunks));
}
"""

display(HTML('<p id="recording-status">Click the cell below to start recording (5 seconds).</p>'))
print("Ready to record. Run the next cell to start a 5-second recording.")

In [ ]:
# Record 5 seconds of audio
display(Javascript(RECORD_JS + 'google.colab.kernel.invokeFunction("notebook.record", [await record(5)], {});'))

import io
import ffmpeg

recorded_audio = None

def save_recording(data):
    global recorded_audio, audio_filename
    binary = b64decode(data.split(',')[1])
    audio_filename = 'recording.wav'
    with open(audio_filename, 'wb') as f:
        f.write(binary)
    print(f"Saved recording to {audio_filename}")
    display(Audio(audio_filename))

output.register_callback('notebook.record', save_recording)

## Analyze the Audio

In [ ]:
results = classifier(audio_filename)

# Find top emotion
top = max(results, key=lambda x: x["score"])
top_label = LABEL_MAP.get(top["label"], top["label"])
top_style = EMOTION_STYLE.get(top_label, {"emoji": "?", "color": "#999"})

# Build HTML output
bars = ''
for r in sorted(results, key=lambda x: x["score"], reverse=True):
    label = LABEL_MAP.get(r["label"], r["label"])
    style = EMOTION_STYLE.get(label, {"emoji": "?", "color": "#999"})
    pct = r["score"] * 100
    bars += f'''
    <div style="display:flex;align-items:center;gap:10px;margin:8px 0;">
        <span style="font-size:24px;">{style["emoji"]}</span>
        <span style="width:70px;font-weight:600;">{label}</span>
        <div style="flex:1;background:#f0f0f0;border-radius:6px;height:24px;overflow:hidden;">
            <div style="width:{pct}%;background:{style["color"]};height:100%;
                        border-radius:6px;min-width:3px;"></div>
        </div>
        <span style="width:45px;text-align:right;color:#888;">{pct:.0f}%</span>
    </div>'''

display(HTML(f'''
<div style="font-family:system-ui;max-width:500px;">
    <div style="text-align:center;padding:20px 0;">
        <span style="font-size:64px;">{top_style["emoji"]}</span>
        <div style="font-size:1.2em;font-weight:700;margin-top:8px;
                    color:{top_style["color"]};">{top_label}</div>
    </div>
    {bars}
</div>
'''))

## Tip

Try saying the same sentence in different emotional tones:
- Say "Hello, how are you?" cheerfully vs. flatly vs. angrily
- The model responds to vocal characteristics (pitch, energy, pace), not word content